1. Carga de librerias

In [ ]:
import torch
import os
import pandas as pd
import random
import numpy as np
from google.colab import drive
from transformers import GPT2Tokenizer
from transformers import GPT2LMHeadModel
from torch.optim import AdamW
from torch.utils.data import DataLoader
from datetime import datetime

Configuración general del modelo

In [ ]:
# CONFIGURACION GENERAL DEL PROYECTO

# Reproducibilidad
seed = 42

# Dataset
train_split = 0.90

# Tokenizacion
expected_vocab_size = 50257

# Secuencias
context_size = 256
batch_size = 16

# Modelo GPT from scratch
embed_dim = 128
num_heads = 4
num_layers = 4
dropout = 0.1

# Entrenamiento
learning_rate = 1e-4
max_epochs = 4
eval_interval = 200

# Dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

print("CONFIGURACION CARGADA")
print(f"Device: {device}")
print(f"Context size: {context_size}")
print(f"Batch size: {batch_size}")
print(f"Learning rate: {learning_rate}")

2. Carga de datos

In [ ]:
drive.mount('/content/drive')
base = '/content/drive/MyDrive/Colab Notebooks/MAIA'
path = f'{base}/data/raw/nlp_llm_mp2/input.txt'

with open(path, 'r') as f:
    corpus = f.read()

print(corpus[:500])

Validación inicial del corpus

In [ ]:
print("VALIDACION INICIAL DEL CORPUS")

print(f"Total de caracteres: {len(corpus):,}")

words = corpus.split()
print(f"Total aproximado de palabras: {len(words):,}")

lines = corpus.splitlines()
print(f"Total de lineas: {len(lines):,}")

print("\nPrimeras 5 lineas:")
for i in range(5):
    print(f"{i+1}: {lines[i]}")

print("\nUltimas 5 lineas:")
for i in range(5):
    print(f"{len(lines)-4+i}: {lines[-5+i]}")

Tokenización del corpus

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

print("VALIDACION DEL TOKENIZER GPT2")

vocab_size = tokenizer.vocab_size
print(f"Tamaño del vocabulario: {vocab_size}")

tokens = tokenizer.encode(
    corpus,
    add_special_tokens=False
)

print(f"Total de tokens del corpus: {len(tokens):,}")

print("\nPrimeros 20 token IDs:")
print(tokens[:20])


Validación del tamaño del corpus

In [ ]:
actual_vocab_size = tokenizer.vocab_size

assert actual_vocab_size == expected_vocab_size

División del conjunto de datos: Train / Validation

In [ ]:
split_idx = int(train_split * len(tokens))

train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

print("DIVISION TRAIN VALIDATION")

print(f"Total de tokens: {len(tokens):,}")
print(f"Tokens de entrenamiento: {len(train_tokens):,}")
print(f"Tokens de validacion: {len(val_tokens):,}")

print(f"\nPorcentaje train: {len(train_tokens)/len(tokens):.2%}")
print(f"Porcentaje validation: {len(val_tokens)/len(tokens):.2%}")

Convertimos a tensor

In [ ]:
train_data = torch.tensor(train_tokens, dtype=torch.long)
val_data = torch.tensor(val_tokens, dtype=torch.long)

print("DATOS CONVERTIDOS A TENSOR")

print(f"Train shape: {train_data.shape}")
print(f"Validation shape: {val_data.shape}")
print(f"Tipo de dato: {train_data.dtype}")

Funcion que realiza el desplazamiento de tokens para el entrenamiento (batches autoregresivos)

In [ ]:
"""Función que realiza el desplazamiento de tokens para el entrenamiento
utilizada para la construcción del modelo desde cero"""

def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - context_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + context_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + context_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [ ]:
x_batch, y_batch = get_batch("train")

print("VALIDACION DE BATCHES")

print(f"x shape: {x_batch.shape}")
print(f"y shape: {y_batch.shape}")

print("\nPrimer ejemplo de entrada (x):")
print(x_batch[0])

print("\nPrimer ejemplo target (y):")
print(y_batch[0])

3. Modelo 1: Fine-tuning GPT2

: Carga de modelo pre-entrenado

In [ ]:
print("CARGA DEL MODELO GPT2")

model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
model_gpt2 = model_gpt2.to(device)

print("Modelo cargado correctamente")
print(f"Device: {device}")

Creación de la clase para construir los dataset para el modelo GPT2 pre-entrenado

In [ ]:
from torch.utils.data import Dataset

class GPT2Dataset(Dataset):
    def __init__(self, tokens, context_size):
        self.input_ids = []

        for i in range(0, len(tokens) - context_size, context_size):
            block = tokens[i:i + context_size]
            self.input_ids.append(torch.tensor(block, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids = self.input_ids[idx]

        return {
            "input_ids": input_ids,
            "labels": input_ids.clone()
        }

Creación de los dataset

In [ ]:
train_dataset = GPT2Dataset(train_tokens, context_size)
val_dataset = GPT2Dataset(val_tokens, context_size)

print("DATASETS GPT2 CREADOS")

print(f"Bloques de entrenamiento: {len(train_dataset):,}")
print(f"Bloques de validacion: {len(val_dataset):,}")

In [ ]:
sample = train_dataset[0]

print("VALIDACION DEL DATASET GPT2")

print(f"Input shape: {sample['input_ids'].shape}")
print(f"Labels shape: {sample['labels'].shape}")

print("\nPrimer bloque input_ids:")
print(sample["input_ids"])

print("\nPrimer bloque labels:")
print(sample["labels"])

Construcción del dataloader

In [ ]:
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DATALOADERS CREADOS")

print(f"Seed utilizada: {seed}")
print(f"Batch size: {batch_size}")
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

In [ ]:
batch = next(iter(train_loader))

print("VALIDACION DEL DATALOADER")

print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"labels shape: {batch['labels'].shape}")

Guardado de modelo (primera vez - creacion del archivo)

In [ ]:
log_path = "experiments_log.csv"

if not os.path.exists(log_path):
    df_log = pd.DataFrame(columns=[
        "version",
        "date",
        "model",
        "context_size",
        "batch_size",
        "learning_rate",
        "max_epochs",
        "train_split",
        "validation_split",
        "device",
        "final_train_loss",
        "best_validation_loss",
        "best_epoch",
        "overfitting_detected",
        "checkpoint_path",
        "notes"
    ])

    df_log.to_csv(log_path, index=False)

print("LOG DE EXPERIMENTOS INICIALIZADO")
print(f"Archivo: {log_path}")

👉 Definición del optimizer y entrenamiento del modelo

In [ ]:

optimizer = AdamW(
    model_gpt2.parameters(),
    lr=learning_rate,
    weight_decay=0.001
)

print("OPTIMIZER CONFIGURADO")
print(f"Learning rate: {learning_rate}")

best_val_loss = float("inf")
best_epoch = 0
final_train_loss = 0
train_loss_at_best_epoch = 0

df_log = pd.read_csv(log_path)
next_version = f"v{len(df_log) + 1}"
checkpoint_path = f"gpt2_finetuned_{next_version}.pt"

for epoch in range(max_epochs):
    model_gpt2.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_gpt2(
            input_ids=input_ids,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    final_train_loss = avg_train_loss

    model_gpt2.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_gpt2(
                input_ids=input_ids,
                labels=labels
            )

            loss = outputs.loss
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        train_loss_at_best_epoch = avg_train_loss

        torch.save(
            model_gpt2.state_dict(),
            checkpoint_path
        )

        print(f"Nuevo mejor modelo guardado en epoch {best_epoch}")

    print(
        f"Epoch {epoch + 1}/{max_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Validation Loss: {avg_val_loss:.4f}"
    )

print("\nMEJOR RESULTADO")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Best Epoch: {best_epoch}")
print(f"Train Loss at Best Epoch: {train_loss_at_best_epoch:.4f}")
print(f"Best Checkpoint: {checkpoint_path}")

Guardado de experimento

In [ ]:
df_log = pd.read_csv(log_path)

next_version = f"v{len(df_log) + 1}"
checkpoint_path = f"gpt2_finetuned_{next_version}.pt"

torch.save(
    model_gpt2.state_dict(),
    checkpoint_path
)

new_experiment = {
    "version": next_version,
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": "GPT2 Fine-Tuning",
    "context_size": context_size,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "max_epochs": max_epochs,
    "train_split": train_split,
    "validation_split": 1 - train_split,
    "device": device,
    "final_train_loss": final_train_loss,
    "best_validation_loss": best_val_loss,
    "best_epoch": best_epoch,
    "overfitting_detected": "Yes" if best_val_loss > train_loss_at_best_epoch else "No",
    "checkpoint_path": checkpoint_path,
    "notes": "Primer experimento estructural sobre longitud de contexto manteniendo baseline optimizado"
}

df_log.loc[len(df_log)] = new_experiment
df_log.to_csv(log_path, index=False)

print("EXPERIMENTO REGISTRADO")
print(df_log.tail(1))

Generación de Texto (Inferencia)

In [ ]:
def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device
):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return generated_text

print("FUNCION DE GENERACION CARGADA")

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

print("PAD TOKEN CONFIGURADO")
print(f"Pad token: {tokenizer.pad_token}")
print(f"Pad token id: {tokenizer.pad_token_id}")

In [ ]:
prompts = [
    "ROMEO:",
    "KING:",
    "To be, or not to be"
]

print("GENERACION DE TEXTO\n")

for i, prompt in enumerate(prompts, 1):
    generated = generate_text(
        model=model_gpt2,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=120,
        temperature=0.8,
        top_k=50
    )

    print(f"Ejemplo {i}")
    print(f"Prompt: {prompt}")
    print(generated)
    print("\n" + "-" * 80 + "\n")

Reinicio de log (reseteo del log de experimentos)

In [ ]:
log_path = "experiments_log.csv"

if os.path.exists(log_path):
    os.remove(log_path)
    print("Archivo de log anterior eliminado")

df_log = pd.DataFrame(columns=[
    "version",
    "date",
    "model",
    "context_size",
    "batch_size",
    "learning_rate",
    "max_epochs",
    "train_split",
    "validation_split",
    "device",
    "final_train_loss",
    "best_validation_loss",
    "best_epoch",
    "overfitting_detected",
    "checkpoint_path",
    "notes"
])

df_log.to_csv(log_path, index=False)

print("LOG REINICIADO CORRECTAMENTE")
print(f"Nuevo archivo creado: {log_path}")